# RotorQuant KV-cache compression

RotorQuant is also a KV-cache method, not a saved weight format. The RotorQuant report replaces dense TurboQuant-style rotations with small Clifford/3D rotor blocks, claiming fewer parameters and faster kernels.

Sources:
- Technical report/site: https://scrya.com/rotorquant
- Code: https://github.com/scrya-com/rotorquant
- TurboQuant paper it builds on: https://arxiv.org/abs/2504.19874

This notebook implements a self-contained 3D-rotor block reference path using Rodrigues rotations. It saves a compressed KV cache artifact and theoretical packed-memory metrics. Use the linked repo for the CUDA/Metal kernels and llama.cpp experiments.

In [ ]:
!pip -q install -U transformers accelerate sentencepiece safetensors

In [ ]:

import json
import math
from pathlib import Path

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM


def bytes_to_mib(n):
    return n / (1024 ** 2)


def random_3d_rotations(n_blocks, device, dtype, seed=5678):
    g = torch.Generator(device=device)
    g.manual_seed(seed)
    axis = torch.randn(n_blocks, 3, generator=g, device=device, dtype=torch.float32)
    axis = axis / axis.norm(dim=-1, keepdim=True).clamp(min=1e-6)
    angle = torch.rand(n_blocks, generator=g, device=device, dtype=torch.float32) * (2 * math.pi)
    kx, ky, kz = axis.unbind(-1)
    c = torch.cos(angle)
    s = torch.sin(angle)
    one_c = 1 - c
    r = torch.zeros(n_blocks, 3, 3, device=device, dtype=torch.float32)
    r[:, 0, 0] = c + kx * kx * one_c
    r[:, 0, 1] = kx * ky * one_c - kz * s
    r[:, 0, 2] = kx * kz * one_c + ky * s
    r[:, 1, 0] = ky * kx * one_c + kz * s
    r[:, 1, 1] = c + ky * ky * one_c
    r[:, 1, 2] = ky * kz * one_c - kx * s
    r[:, 2, 0] = kz * kx * one_c - ky * s
    r[:, 2, 1] = kz * ky * one_c + kx * s
    r[:, 2, 2] = c + kz * kz * one_c
    return r.to(device=device, dtype=dtype)


def apply_block_rotations(x, rotations, inverse=False):
    orig_shape = x.shape
    dim = orig_shape[-1]
    pad = (-dim) % 3
    if pad:
        x = F.pad(x, (0, pad))
    blocks = x.reshape(*x.shape[:-1], -1, 3)
    r = rotations
    if inverse:
        r = r.transpose(-1, -2)
    y = torch.einsum("...bi,bij->...bj", blocks, r)
    y = y.reshape(*x.shape[:-2], -1)
    if pad:
        y = y[..., :dim]
    return y.reshape(orig_shape)


def quantize_rotor_vectors(x, bits, rotations):
    rotated = apply_block_rotations(x, rotations, inverse=False)
    orig_shape = rotated.shape
    dim = orig_shape[-1]
    flat = rotated.reshape(-1, dim)
    qmax = (2 ** (bits - 1)) - 1
    scale = flat.abs().amax(dim=-1, keepdim=True).clamp(min=1e-6) / qmax
    q = torch.round(flat / scale).clamp(-qmax, qmax).to(torch.int8)
    deq_rotated = (q.to(rotated.dtype) * scale).reshape(orig_shape)
    deq = apply_block_rotations(deq_rotated, rotations, inverse=True)
    return {"q": q.cpu(), "scale": scale.cpu().to(torch.float16), "dequantized": deq.to(x.dtype)}


def kv_cache_bytes(past_key_values):
    total = 0
    for key, value in past_key_values:
        total += key.numel() * key.element_size()
        total += value.numel() * value.element_size()
    return total


def cosine_report(original, reconstructed):
    a = original.reshape(-1, original.shape[-1]).float()
    b = reconstructed.reshape(-1, reconstructed.shape[-1]).float()
    cos = F.cosine_similarity(a, b, dim=-1)
    mse = F.mse_loss(b, a).item()
    return float(cos.mean().item()), float(cos.min().item()), float(mse)


def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(json.dumps(payload, indent=2))


In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = Path("/content/quant_outputs/rotorquant_kv")
PROMPT = "RotorQuant compresses the key value cache using compact block rotations. " * 32
BITS = 3

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
).to(device)
model.eval()

inputs = tokenizer(PROMPT, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs, use_cache=True)
past = outputs.past_key_values

head_dim = past[0][0].shape[-1]
n_blocks = math.ceil(head_dim / 3)
rotations = random_3d_rotations(n_blocks, device, past[0][0].dtype)

artifact = {"method": "rotorquant_reference", "bits": BITS, "layers": []}
reports = []
packed_bits = 0
for layer_idx, (key, value) in enumerate(past):
    qk = quantize_rotor_vectors(key, BITS, rotations)
    qv = quantize_rotor_vectors(value, BITS, rotations)
    k_mean, k_min, k_mse = cosine_report(key, qk["dequantized"].to(device))
    v_mean, v_min, v_mse = cosine_report(value, qv["dequantized"].to(device))
    n_values = key.numel() + value.numel()
    n_vectors = key.reshape(-1, head_dim).shape[0] + value.reshape(-1, head_dim).shape[0]
    packed_bits += n_values * BITS
    packed_bits += n_vectors * 16  # per-vector fp16 scale
    artifact["layers"].append({
        "key_q": qk["q"],
        "key_scale": qk["scale"],
        "value_q": qv["q"],
        "value_scale": qv["scale"],
    })
    reports.append({
        "layer": layer_idx,
        "key_cosine_mean": k_mean,
        "key_cosine_min": k_min,
        "key_mse": k_mse,
        "value_cosine_mean": v_mean,
        "value_cosine_min": v_min,
        "value_mse": v_mse,
    })

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(artifact, OUTPUT_DIR / "compressed_kv_cache.pt")

fp_bytes = kv_cache_bytes(past)
packed_bytes = math.ceil(packed_bits / 8)
metrics = {
    "method": "rotorquant_reference_kv_cache",
    "model_id": MODEL_ID,
    "bits": BITS,
    "fp_cache_mib": bytes_to_mib(fp_bytes),
    "theoretical_packed_cache_mib": bytes_to_mib(packed_bytes),
    "compression_ratio": fp_bytes / packed_bytes,
    "artifact_path": str(OUTPUT_DIR / "compressed_kv_cache.pt"),
    "reports": reports,
}
save_json(OUTPUT_DIR / "metrics.json", metrics)

In [ ]:
# Optional: clone RotorQuant for its native benchmark and kernel path.
# !git clone https://github.com/scrya-com/rotorquant /content/rotorquant
# %cd /content/rotorquant
# !python -m turboquant.benchmark_perplexity --bits 3 4